In [1]:
!pip install -q transformers < 5.0.0

/bin/bash: line 1: 5.0.0: No such file or directory


In [2]:
import torch
from transformers import pipeline

# GPU/CPU 자동 감지
device = 0 if torch.cuda.is_available() else -1

# 모델을 직접 지정하여 파이프라인 생성
classifier = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=device
)

result = classifier("I love programming!")
print(result)
# [{'label': 'POSITIVE', 'score': 0.9998}]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9998701810836792}]


**2-3 한국어 감성 분석**

In [3]:
## ko_sentiment_compare.py -1

import torch
from transformers import pipeline

# GPU/CPU 자동 감지
device = 0 if torch.cuda.is_available() else -1

# ── 모델 1: 다국어 모델 (별점 1~5) ──
multi_clf = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    device=device
)

# ── 모델 2: 한국어 감성 분석 전용 ──.
    "sentiment-analysis",
    model="matthewburke/korean_sentiment",
    device=device
)

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [4]:
## ko_sentiment_compare.py -2

texts = [
    "이 제품 정말 좋아요! 강력 추천합니다.",
    "배송이 너무 늦고 포장도 엉망이었어요.",
    "가격은 괜찮은데 품질이 아쉬워요.",
]

print("=" * 60)
print("다국어 모델 vs 한국어 전용 모델 비교")
print("=" * 60)

for text in texts:
    r1 = multi_clf(text)[0]
    r2 = ko_clf(text)[0]
    print(f"\n입력: \"{text}\"")
    print(f"  다국어 모델: {r1['label']:>10s} ({r1['score']:.4f})")
    print(f"  한국어 전용: {r2['label']:>10s} ({r2['score']:.4f})")

다국어 모델 vs 한국어 전용 모델 비교

입력: "이 제품 정말 좋아요! 강력 추천합니다."
  다국어 모델:    5 stars (0.7265)
  한국어 전용:    LABEL_1 (0.9728)

입력: "배송이 너무 늦고 포장도 엉망이었어요."
  다국어 모델:    2 stars (0.4615)
  한국어 전용:    LABEL_0 (0.9591)

입력: "가격은 괜찮은데 품질이 아쉬워요."
  다국어 모델:    2 stars (0.4764)
  한국어 전용:    LABEL_0 (0.9328)


**2-4 한국어 번역**

1단계: GPU 런타임으로 변경

  상단 메뉴 → 런타임 → 런타임 유형 변경 → T4 GPU 선택 → 저장

2단계: 런타임 재시작 후 다시 실행

  코드는 그대로 사용하면 됩니다. 단, GPU 연결 전에 확인하는 코드를 맨 위에 추가하면 좋습니다.

In [5]:
import torch
print(torch.cuda.is_available())       # True여야 정상
print(torch.cuda.get_device_name(0))   # ex) Tesla T4
device = "cuda" if torch.cuda.is_available() else "cpu"

True
Tesla T4


In [6]:
import torch
from transformers import MarianMTModel, MarianTokenizer

def translate(text, model_name):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = MarianTokenizer.from_pretrained(model_name)
    model = MarianMTModel.from_pretrained(model_name).to(device)
    inputs = tokenizer([text], return_tensors="pt", padding=True).to(device)
    translated = model.generate(**inputs)
    return tokenizer.decode(translated[0], skip_special_tokens=True)

# 한→영: opus-mt-ko-en (존재 O)
print("한→영:", translate("오늘 날씨가 정말 좋습니다.", "Helsinki-NLP/opus-mt-ko-en"))

# 영→한: opus-mt-tc-big-en-ko (존재 O) ← 이걸 써야 함
print("영→한:", translate("I love learning artificial intelligence!", "Helsinki-NLP/opus-mt-tc-big-en-ko"))

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/842k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/813k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/418M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

영→한: US 92년 전


**2-5 한국어 텍스트 요약**

In [7]:
import torch
from transformers import PreTrainedTokenizerFast, BartForConditionalGeneration

# ── 1단계: Tokenizer + Model 직접 로드 ──
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = PreTrainedTokenizerFast.from_pretrained("gogamza/kobart-summarization")
model = BartForConditionalGeneration.from_pretrained("gogamza/kobart-summarization").to(device)

article = """
인공지능(AI) 기술이 다양한 산업에 빠르게 확산되고 있다. 의료 분야에서는 AI를 활용한
질병 진단 시스템이 의사의 판단을 보조하고 있으며, 금융 분야에서는 이상 거래 탐지와
신용 평가에 AI가 도입되었다. 교육 분야에서도 개인 맞춤형 학습 시스템이 등장하여
학생들의 학습 효율을 높이고 있다. 전문가들은 앞으로 AI가 더 많은 직업에 영향을
미칠 것으로 전망하면서도, 윤리적 문제와 일자리 변화에 대한 사회적 논의가 필요하다고
강조하고 있다.
"""

# ── 2단계: 토큰화 (텍스트 → 숫자) ──
inputs = tokenizer([article], return_tensors="pt", max_length=1024, truncation=True).to(device)

# ── 3단계: 모델 추론 (요약 생성) ──
summary_ids = model.generate(inputs["input_ids"], max_length=80, min_length=20, num_beams=4)

# ── 4단계: 후처리 (숫자 → 텍스트) ──
result = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print("요약 결과:", result)

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/4.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

요약 결과: 인공지능인공지능(AI) 기술이 의료 분야에서는 AI를 활용한 의료 분야에서는 AI를 활용한 AI를 활용한 진료질병 진단 시스템이 의사의 판단을 보조하고 있으며, 금융 분야에서는 이상 거래 탐지와 IB신용 평가에 AI가 도입되었다.


**3-2 제로샷 분류**

In [8]:
import torch
from transformers import pipeline

# GPU/CPU 자동 감지
device = 0 if torch.cuda.is_available() else -1

zero_classifier = pipeline("zero-shot-classification", device=device)

text = "The new iPhone 16 has an amazing camera with AI features."

# 분류할 카테고리를 자유롭게 지정
result = zero_classifier(
    text,
    candidate_labels=["technology", "sports", "politics", "entertainment"]
)

print(f"텍스트: {text}\n")
for label, score in zip(result['labels'], result['scores']):
    bar = "█" * int(score * 30)
    print(f"  {label:15s} {score:.4f} {bar}")

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

텍스트: The new iPhone 16 has an amazing camera with AI features.

  technology      0.9734 █████████████████████████████
  entertainment   0.0172 
  sports          0.0057 
  politics        0.0037 


In [9]:
result = zero_classifier(
    "i got a gift",
    candidate_labels=["happy", "sad", "angry", "surprised"]
)

print("영어 ",result)
print("=" * 60)
# 실험 2: 한국어 텍스트 + 영어 카테고리
result = zero_classifier(
    "삼성전자가 새로운 AI 칩을 발표했습니다.",
    candidate_labels=["technology", "sports", "politics", "entertainment"]
)

print("한국 + 영어 카테고리 ",result)

영어  {'sequence': 'i got a gift', 'labels': ['surprised', 'happy', 'sad', 'angry'], 'scores': [0.6625829935073853, 0.33017927408218384, 0.004500498063862324, 0.0027372268959879875]}
한국 + 영어 카테고리  {'sequence': '삼성전자가 새로운 AI 칩을 발표했습니다.', 'labels': ['technology', 'entertainment', 'sports', 'politics'], 'scores': [0.695780873298645, 0.16464918851852417, 0.10021476447582245, 0.03935513272881508]}
